# Transformation des données

In [1]:
%pip -q install -U ipykernel
%pip -q install pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
path_in   = "M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/base/"
path_load = "M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/"
path_prep = "M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/prep_data/"  

In [3]:
import pandas as pd
from datetime import datetime

# Nettoyage des dates 

In [4]:


def parse_date_with_time_range(date_str):
    if pd.isna(date_str):
        return "unknown"
    
    date_str = str(date_str).strip()
    
    # Supprimer la partie horaire
    if '—' in date_str:
        date_part = date_str.split('—')[0].strip()
    else:
        date_part = date_str
    
    # Gérer les plages de dates
    if '–' in date_part:
        start, end = date_part.split('–')
        start = start.strip()
        end = end.strip()
        
        # Si la date de début n'inclut pas le mois ou l'année, les ajouter depuis la date de fin
        if len(start.split()) == 1:
            start = f"{start} {' '.join(end.split()[1:])}"
        elif len(start.split()) == 2:
            start = f"{start} {end.split()[-1]}"
        
        try:
            start_date = datetime.strptime(start, "%d %B %Y").strftime("%Y-%m-%d")
            end_date = datetime.strptime(end, "%d %B %Y").strftime("%Y-%m-%d")
            return f"{start_date} to {end_date}"
        except ValueError:
            return date_str
    
    # Gérer les dates simples
    try:
        return datetime.strptime(date_part, "%d %B %Y").strftime("%Y-%m-%d")
    except ValueError:
        return date_str

In [5]:
from datetime import datetime
import pandas as pd

def standardize_single_date(date_str, year=None):
    if pd.isna(date_str) or date_str == '—':
        return "unknown"
    
    date_str = str(date_str).strip()
    
    try:
        if len(date_str.split()) == 2:  # Si seulement jour et mois
            date_str = f"{date_str} {year}"
        return datetime.strptime(date_str, "%d %B %Y").strftime("%Y-%m-%d")
    except ValueError:
        return "unknown"

def standardize_date_range(date_str, year):
    if pd.isna(date_str) or date_str == '—':
        return "unknown", "unknown"
    
    if '–' in date_str:
        start, end = date_str.split('–')
        start_date = standardize_single_date(start.strip(), year)
        end_date = standardize_single_date(end.strip(), year)
        return start_date, end_date
    return "unknown", "unknown"

def process_olympic_dates(row):
    year = row['year']
    
    # Traiter d'abord competition_date
    comp_start, comp_end = standardize_date_range(row['competition_date'], year)
    
    # Standardiser et compléter start_date
    row['start_date'] = standardize_single_date(row['start_date'], year)
    if row['start_date'] == "unknown" and comp_start != "unknown":
        row['start_date'] = comp_start
        
    # Standardiser et compléter end_date
    row['end_date'] = standardize_single_date(row['end_date'], year)
    if row['end_date'] == "unknown" and comp_end != "unknown":
        row['end_date'] = comp_end
    
    # Mettre à jour competition_date au format standardisé
    if row['start_date'] != "unknown" and row['end_date'] != "unknown":
        row['competition_date'] = f"{row['start_date']} to {row['end_date']}"
    
    return row

In [6]:
data_0 = pd.read_csv(path_in + 'Olympics_Games.csv')
data_1 = pd.read_csv(path_in + 'Olympic_Athlete_Event_Results.csv')
data_2 = pd.read_csv(path_in + 'Olympic_Games_Medal_Tally.csv')
data_3 = pd.read_csv(path_in + 'Olympic_Results.csv')

# Format edition_id 
data_0['edition_id'] = 'ed_' + data_0['edition_id'].astype(str)  
data_1['edition_id'] = 'ed_' + data_1['edition_id'].astype(str) 
data_2['edition_id'] = 'ed_' + data_2['edition_id'].astype(str)       
data_3['edition_id'] = 'ed_' + data_3['edition_id'].astype(str) 

# Sauvegarder dans un nouveau fichier
data_0.to_csv(path_prep + 'Olympics_Games_Standardized.csv', index=False)
data_1.to_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv', index=False)
data_2.to_csv(path_prep + 'Olympic_Games_Medal_Tally_Standardized.csv', index=False)
data_3.to_csv(path_prep + 'Olympic_Results_Standardized.csv', index=False)

In [7]:
# Application du traitement de standardisation des dates
data = pd.read_csv(path_in + 'Olympics_Games.csv')

data = data.apply(process_olympic_dates, axis=1)
# Sauvegarde du résultat
data.to_csv(path_prep + 'Olympics_Games_Standardized.csv', index=False)

In [8]:
data = pd.read_csv(path_in + 'Olympic_Athlete_Bio.csv')

for column in ['born']:
    data[column] = data[column].fillna('unknown').apply(parse_date_with_time_range) 
# Sauvegarder dans un nouveau fichier
data.to_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv', index=False)

In [9]:
data = pd.read_csv(path_prep + 'Olympic_Results_Standardized.csv')

for column in ['result_date']:
    data[column] = data[column].fillna('unknown').apply(parse_date_with_time_range)   
# Sauvegarder dans un nouveau fichier
data.to_csv(path_prep + 'Olympic_Results_Standardized.csv', index=False)

In [10]:
olympics_country = pd.read_csv(path_in + 'Olympics_Country.csv')
if 'UNK' not in olympics_country['noc'].values:
    # Créer un nouveau DataFrame avec l'enregistrement à ajouter
    new_row = pd.DataFrame({'noc': ['UNK'], 'country': ['UNKNOWN']})
    # Concaténer le nouveau DataFrame avec olympics_country
    olympics_country = pd.concat([olympics_country, new_row], ignore_index=True) 
# Sauvegarder dans un nouveau fichier
olympics_country.to_csv(path_prep + 'Olympics_Country_Standardized.csv', index=False)

In [11]:
import re
# exract a string r'\b(Summer)\b'
# extract year r'\b(\d{4})\b'
# year = extract_pattern("2019 Summer Olympics", r'\b(\d{4})\b')
def extract_pattern(string, pattern):
    match = re.search(pattern, string)
    if match:
        return match.group(1)
    return None

def calculate_age_participation(row):
    edition = extract_pattern(row['edition'],r'\b(\d{4})\b')
    if pd.isna(row['born']) or pd.isna(edition):
#        print(f" date_of_birth: {row['born']} edition : {row['edition_x']}")
        return 'unknown'
    else:
        try:
            birth_year = pd.to_datetime(row['born']).year
            age = int(edition) - int(birth_year)
#            print(f"{row['athlete']} : birth_year {birth_year}  : edition : {edition}  :  age : {age}")
            return age
        except:
            return 'unknown'

LEFT JOIN Olympic_Athlete_Event_Results.csv file with Olympic_Athlete_Bio.csv (on
athlete_id) for complete information of the athlete (i.e., height, weight, date of birth
when participating in the event)

In [12]:
from datetime import datetime

# Charger les fichiers CSV
athlete_events = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')
results = pd.read_csv(path_prep + 'Olympic_Results_Standardized.csv')

# Première jointure pour les informations des athlètes
df = pd.merge(
    athlete_events,
    results,
    on='result_id',
    how='left'
)
#print(df.columns)
df['edition'] = df['edition_x'].fillna(df['edition_y'])
df = df.drop(['edition_x', 'edition_y'], axis=1)

df['sport'] = df['sport_x'].fillna(df['sport_y'])
df = df.drop(['sport_x', 'sport_y'], axis=1)

df['edition_id'] = df['edition_id_x'].fillna(df['edition_id_y'])
df = df.drop(['edition_id_x', 'edition_id_y'], axis=1)

columns=['edition', 'edition_id', 'result_id','athlete','athlete_id',
         'country_noc','sport','event','pos','medal', 'sport_url','result_date', 
         'result_location', 'result_participants','result_format', 'result_detail', 'result_description']

df = df[columns]
df.to_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv', index=False)
print(df.columns)

Index(['edition', 'edition_id', 'result_id', 'athlete', 'athlete_id',
       'country_noc', 'sport', 'event', 'pos', 'medal', 'sport_url',
       'result_date', 'result_location', 'result_participants',
       'result_format', 'result_detail', 'result_description'],
      dtype='object')


In [13]:
from datetime import datetime

# Charger les fichiers CSV
athlete_events = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')
athlete_bio = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv')

# Première jointure pour les informations des athlètes
df = pd.merge(
    athlete_events,
    athlete_bio,
    on='athlete_id',
    how='left'
)
df['country_noc'] = df['country_noc_x'].fillna(df['country_noc_y'])
df = df.drop(['country_noc_x', 'country_noc_y'], axis=1)

#print(df.columns)
# Extraire l'année et la saison de edition_x
df['year'] = df['edition'].str.extract(r'(\d{4})')
df['season'] = df['edition'].apply(lambda x: extract_pattern(x, r'\d{4}\s+(\w+)\s+Olympics'))

# Calcul du BMI
# Convertir les colonnes en numérique
df['weight'] = pd.to_numeric(df['weight'], errors='coerce')
df['height'] = pd.to_numeric(df['height'], errors='coerce')

# Calculer le BMI après conversion
df['bmi'] = df['weight'] / ((df['height']/100) ** 2)

# Identifier les colonnes avec des données manquantes
missing_values  = df.isnull().sum()
missing_columns = missing_values[missing_values > 0]

# Calcul de l'âge
df['age_participation'] = df.apply(calculate_age_participation, axis=1)

columns=['edition', 'edition_id','athlete','athlete_id','sex','born','country_noc',
         'country','age_participation','bmi','height','weight','sport','event',
         'result_id','pos','medal', 'description', 'special_notes']
df = df[columns]

df.to_csv(path_prep + 'Olympic_Athlete_Bio_Prepared.csv', index=False)
print(df.columns)

Index(['edition', 'edition_id', 'athlete', 'athlete_id', 'sex', 'born',
       'country_noc', 'country', 'age_participation', 'bmi', 'height',
       'weight', 'sport', 'event', 'result_id', 'pos', 'medal', 'description',
       'special_notes'],
      dtype='object')


<p style="text-align: center">
<img src="images/model.jpeg" alt="Olympics Games" width=600 large=450/>
</p>

In [14]:
# Définit le noeud et les données ATHLETE une entrée par athlete_id avec le premier nom ne tient pas compte des mariages
def process_athlete(distinct_athletes):    
    with \
         open(path_load + "fichiers/" + 'borned_in.csv','w', newline='') as _borned_in_file, \
         open(path_load + "fichiers/" + 'athlete.csv', 'w', newline='') as _athlete_file:  

         _athlete_file.write(f"athlete_id:ID,name,sex, :LABEL\n")    
         _borned_in_file.write(f":START_ID,:END_ID, :TYPE\n")
         for _, row in distinct_athletes.iterrows():
            name = row['athlete'].strip().replace(',', '.')
            sex  = row['sex']
            athlete_id = row['athlete_id']
            _athlete_file.write(f"{athlete_id},{name}, {sex}, ATHLETE\n")
            _borned_in_file.write(f"{row['athlete_id']}, {row['country_noc'].strip('')}  , BORNED_IN\n")

athlete_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Prepared.csv')
distinct_athletes = athlete_results[['athlete_id', 'athlete','sex','country_noc']].drop_duplicates(subset=['athlete_id'], keep='first').reset_index(drop=True)  

#process_athlete(distinct_athletes)

In [15]:
# Définit le noeud et les données EDITION  
def process_edition(distinct_editions):    
    with \
        open(path_load + "fichiers/"  + 'edition.csv','w', newline='') as _edition_file: 

        _edition_file.write(f"edition_id:ID,edition, :LABEL\n")     
        for _, row in distinct_editions.iterrows():
            edition     = row['edition'].strip().replace(',', '.')
            edition_id  = row['edition_id'].strip() 
            _edition_file.write(f"{edition_id}, {edition}, EDITION\n")

games_medal_tally  = pd.read_csv(path_prep + 'Olympic_Games_Medal_Tally_Standardized.csv')
distinct_editions  = games_medal_tally[['edition_id', 'edition']].drop_duplicates().reset_index(drop=True)
#process_edition(distinct_editions)

In [16]:
# Définit le noeud et les données COUNTRY 
def process_country(distinct_country):    
    with \
        open(path_load + "fichiers/"  + 'country.csv', 'w', newline='') as _country_file:

        _country_file.write(f"country_id:ID,country, :LABEL\n")
        for _, value in distinct_country.iterrows():
            country_noc = value['noc'].strip() 
            country     = value['country'].strip().replace(',', '.')
            _country_file.write(f"{country_noc},{country},COUNTRY\n")

olympics_country = pd.read_csv(path_prep + 'Olympics_Country_Standardized.csv')
distinct_country = olympics_country[['noc','country']].drop_duplicates().reset_index(drop=True)
#process_country(distinct_country)

In [45]:
# Définit le noeud et les données CITY et la relation LOCATED_IN entre CITY et COUNTRY
def process_city(distinct_city):    
    with \
        open(path_load + "fichiers/"  + 'city.csv', 'w', newline='') as _city_file:

        _city_file.write(f'city_id:ID,city, :LABEL\n')
        for _, value in distinct_city.items():
            city_id     = city_id_map.get(value)
            _city_file.write(f'{city_id}, {value}, CITY\n')

olympics_games     = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')
distinct_city      = olympics_games['city'].drop_duplicates().reset_index(drop=True)
city_id_map        = {value: f'ci{i+1}' for i, value in distinct_city.items()}
#process_city(distinct_city)

In [49]:
# Définit le noeud et les données CITY et la relation LOCATED_IN entre CITY et COUNTRY
def process_cityRelShip(distinct_cityCntry):    
    with \
        open(path_load + "fichiers/"  + 'located_in.csv', 'w', newline='') as _located_in_file, \
        open(path_load + "fichiers/"  + 'organized_by.csv','w', newline='') as _organize_by_file:

        _located_in_file.write(f':START_ID,:END_ID, :TYPE\n')
        _organize_by_file.write(f':START_ID,:END_ID, :TYPE\n') 

        for _, row in distinct_cityCntry.iterrows():
            edition_id        = row['edition_id']
            city        = city_edID_map.get(edition_id)
            city_id     = city_id_map.get(city)
            country_noc = row['country_noc'].strip() 
            
            _located_in_file.write(f'{city_id}, {country_noc}, LOCATED_IN\n')
            _organize_by_file.write(f'{edition_id}, {city_id}, ORGANIZED_BY\n')    

olympics_games     = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')

distinct_city      = olympics_games['city'].drop_duplicates().reset_index(drop=True)
city_id_map        = {value: f'ci{i+1}' for i, value in distinct_city.items()}

distinct_cted      = olympics_games[['city','edition_id']].drop_duplicates().reset_index(drop=True)
city_edID_map      = { value['edition_id'] : value['city'] for _, value in distinct_cted.iterrows()}

distinct_edyr      = olympics_games[['edition_id','year']].drop_duplicates().reset_index(drop=True)
year_edID_map      = {value['edition_id'] : value['year'] for _, value in distinct_edyr.iterrows()}

distinct_cityCntry = olympics_games[['edition_id','country_noc']].drop_duplicates().reset_index(drop=True)
#process_cityRelShip(distinct_cityCntry)

In [18]:
 # Définit le noeud MEDAILLE et les données ainsi que la relation WINS entre ATHLETE et MEDAILLE 
def process_medal(olympic_results):    
    with \
        open(path_load + "fichiers/"  + 'medal.csv',           'w', newline='') as _medal_file, \
        open(path_load + "fichiers/"  + 'wins.csv',            'w', newline='') as _wins_file:  

        _medal_file.write(f"medal_id:ID,country_noc,medal, :LABEL\n") 
        _wins_file.write(f":START_ID,:END_ID, :TYPE\n")    

        for i, row in olympic_results.iterrows():
            medal_val    = row['medal']
            country_noc  = row['country_noc'].strip() 
            athlete_id   = row['athlete_id']
            medal_id     = f'md{i+1}'

            if pd.isna(medal_val) or medal_val.lower() not in ["gold", "silver", "bronze"]:
                i -= 1  
            else :      
                _medal_file.write(f"{medal_id}, {country_noc}, {medal_val}, MEDAL\n") 
                _wins_file.write(f"{athlete_id}, {medal_id}, WINS\n")

olympic_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv')
#process_medal(olympic_results)

In [19]:

# Définit le noeud SPORT et EVENT ainsi que la relation PART_OF entre SPORT et EVENT
def process_sport(distinct_events):  
    with \
        open(path_load + "fichiers/"  + 'sport.csv',           'w', newline='') as _sport_file:

        _sport_file.write(f'sport_id:ID,sport, :LABEL\n') 
        for _, row in distinct_events.iterrows():
            sport      = row['sport']
            sport_id   = sport_id_map.get(sport)  
            _sport_file.write(f'{sport_id}, {sport}, SPORT\n')

olympics_res_Std  = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv') 
distinct_sports   = olympics_res_Std.drop_duplicates('sport').apply(lambda col: col.map(lambda x: x.strip().replace(',', '.') if isinstance(x, str) else x))
sport_id_map      = {row['sport']: f'sp{i+1}' for i, row in distinct_sports.iterrows()}
#process_sport(distinct_sports)

In [20]:

# Définit le noeud SPORT et EVENT ainsi que la relation PART_OF entre SPORT et EVENT
def process_event(distinct_events):  
    with \
        open(path_load + "fichiers/"  + 'event.csv',           'w', newline='') as _event_file, \
        open(path_load + "fichiers/"  + 'part_of.csv',         'w', newline='') as _part_of_file, \
        open(path_load + "fichiers/"  + 'sp_cpofevt.csv', 'w', newline='') as _sp_cpofevt_file, \
        open(path_load + "fichiers/"  + 'ed_orgzevt.csv', 'w', newline='') as _ed_orgzevt_file:

        _event_file.write(f'event_id:ID,result_id,event, :LABEL\n')
        _part_of_file.write(f':START_ID,:END_ID, :TYPE\n') 
        _sp_cpofevt_file.write(f":START_ID,:END_ID, :TYPE\n")
        _ed_orgzevt_file.write(f":START_ID,:END_ID, :TYPE\n")    

        for _, row in distinct_events.iterrows():
            result_id  = row['result_id']
            event_id   = event_id_map.get(result_id)
            edition_id = row['edition_id']
            event      = row['event'].strip().replace(',', '.')
            sport      = event_sport_map[event]
            sport_id   = sport_id_map.get(sport)  

            _event_file.write(f'{event_id}, {result_id}, {event}, EVENT\n')
            _part_of_file.write(f'{event_id}, {sport_id}, PART_OF\n') 
            _sp_cpofevt_file.write(f"{sport_id}, {event_id}, SP_CPOFEVT\n")
            _ed_orgzevt_file.write(f"{edition_id}, {event_id}, ED_ORGZEVT\n")

olympics_res_Std = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv') 
distinct_events  = olympics_res_Std.drop_duplicates('result_id').reset_index(drop=True)

distinctEvents   = distinct_events.drop_duplicates('event').apply(lambda col: col.map(lambda x: x.strip().replace(',', '.') if isinstance(x, str) else x))
distinct_sports   = olympics_res_Std.drop_duplicates('sport').apply(lambda col: col.map(lambda x: x.strip().replace(',', '.') if isinstance(x, str) else x))

sport_id_map      = {row['sport']: f'sp{i+1}' for i, row in distinct_sports.iterrows()}
event_id_map     = {row['result_id']: f'ev{i+1}' for i, row in distinct_events.iterrows()}
event_sport_map  =  {row['event']: row['sport'] for i, row in distinctEvents.iterrows()}
#process_event(distinct_events)

In [56]:
# Définit le noeud RESULT et les données associées
def process_result(complete_result):  
    with \
        open(path_load + "fichiers/"  + 'result.csv',          'w', newline='') as _result_to_file, \
        open(path_load + "fichiers/"  + 'performed_in.csv',    'w', newline='') as _performed_in_file, \
        open(path_load + "fichiers/"  + 'competed_in.csv',     'w', newline='') as _competed_in_file:

        _result_to_file.write(f"result_id:ID,event_id, :LABEL\n")            
        _performed_in_file.write(f":START_ID,:END_ID, :TYPE\n")
        _competed_in_file.write(f":START_ID,:END_ID, :TYPE\n") 
        for i, row in complete_result.iterrows():
            restat_id  = f'rs{i+1}'
            result_id  = row['result_id']
            athlete_id = row['athlete_id']
            event_id   = event_id_map.get(result_id)
            _result_to_file.write(f"{restat_id}, {result_id}, RESULT\n")            
            _performed_in_file.write(f"{athlete_id}, {restat_id}, PERFORMED_IN\n")
            _competed_in_file.write(f" {restat_id}, {event_id}, COMPETED_IN\n") 
olympic_res     = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')
distinct_events = olympics_res_Std.drop_duplicates('result_id').reset_index(drop=True) 
event_id_map    = {row['result_id']: f'ev{i+1}' for i, row in distinct_events.iterrows()}

complete_result = olympic_results[['result_id','athlete_id']].drop_duplicates().reset_index(drop=True) 
#process_result(complete_result)      

In [22]:
# Defniri les relation manquantes 
def process_relationship(constest_result):
    with \
        open(path_load + "fichiers/"  + 'represent_to.csv',    'w', newline='') as _represent_to_file, \
        open(path_load + "fichiers/"  + 'edcomp_ath.csv', 'w', newline='') as _edcomp_ath_file, \
        open(path_load + "fichiers/"  + 'athpart_ed.csv', 'w', newline='') as _athpart_ed_file:

        _edcomp_ath_file.write(f":START_ID,:END_ID,:TYPE\n")
        _represent_to_file.write(f":START_ID,EDITION_ID,:END_ID,:TYPE\n")
        _athpart_ed_file.write(f":START_ID,:END_ID,:TYPE\n") 
        for i, row in constest_result.iterrows():
            # ATHLETE COUNTRY
            athlete_id  = row['athlete_id']
            edition_id  = row['edition_id']
            country_noc = row['country_noc'] 
            _represent_to_file.write(f" {athlete_id}, {edition_id}, {country_noc}, REPRESENTS\n")
            # ATHLETE EDITION
            _edcomp_ath_file.write(f" {edition_id}, {athlete_id}, EDCOMP_ATH\n")
            _athpart_ed_file.write(f" {athlete_id}, {edition_id}, ATHPART_ED\n") 

olympics_prepr     = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv')
constest_result  = olympics_prepr[['edition_id', 'country_noc', 'athlete_id']].drop_duplicates().reset_index(drop=True)
#process_relationship(constest_result)

In [23]:
# Créer les fichiers CSV pour les nœuds et les relations
def generate_header():
        headers = {
                #####################################################################################
                'city_header.csv'                      : 'city_id:ID,city, :LABEL',
                'country_header.csv'                   : 'country_id:ID,country, :LABEL',
                'located_in_header.csv'                : ':START_ID,:END_ID, :TYPE',
                #####################################################################################
                'edition_header.csv'                   : 'edition_id:ID,edition, :LABEL',
                'organized_by_header.csv'              : ':START_ID,:END_ID, :TYPE',
                #####################################################################################
                'sport_header.csv'                     : 'sport_id:ID,sport, :LABEL',
                'event_header.csv'                     : 'event_id:ID,result_id,event, :LABEL', 
                'part_of_header.csv'                   : ':START_ID,:END_ID, :TYPE',   
                #####################################################################################
                'athlete_header.csv'                   : 'athlete_id:ID,name,gender, :LABEL',
                'competed_in_header.csv'               : ':START_ID,:END_ID, :TYPE',
                'borned_in_header.csv'                 : ':START_ID,:END_ID, :TYPE',        
                ####################################################################################
                'medal_header.csv'                     : 'medal_id:ID,country_noc,medal, :LABEL',
                'wins_header.csv'                      : ':START_ID,:END_ID, :TYPE',
                #####################################################################################
                'result_header.csv'                    : 'result_id:ID,event_id, :LABEL',
                'represent_to_header.csv'              : ':START_ID,:END_ID,edition_id, :TYPE', 
                #####################################################################################
                'performed_in_header.csv'              : ':START_ID,:END_ID, :TYPE',
                'edcomp_ath_header.csv'                : ':START_ID,:END_ID, :TYPE',
                'athpart_ed_header.csv'                : ':START_ID,:END_ID, :TYPE',
                'sp_cpofevt_header.csv'                : ':START_ID,:END_ID, :TYPE',
                'ed_orgzevt_header.csv'                : ':START_ID,:END_ID, :TYPE',
}        

        for filename, content in headers.items():
                with open(path_load + "fichiers/" + filename, 'w') as f:
                     f.write(content)
        print("Tous les fichiers CSV et leurs fichiers d'en-tête ont été créés avec succès.")
#generate_header()

In [50]:
process_athlete(distinct_athletes)
process_edition(distinct_editions)
process_country(distinct_country)
process_city(distinct_city)
process_cityRelShip(distinct_cityCntry)
process_medal(olympic_results)
process_sport(distinct_sports)
process_event(distinct_events)
process_result(complete_result)  
process_relationship(constest_result)
generate_header()

le tab,  []
Tous les fichiers CSV et leurs fichiers d'en-tête ont été créés avec succès.


<p style="text-align: center">
<img src="images/model.jpeg" alt="Olympics Games" width=600 large=450/>
</p>

In [25]:
'''
bin/neo4j-admin database import full --delimiter="," --quote="'" 
   --nodes=ATHLETE=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/athlete_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/athlete.csv 
   --nodes=EDITION=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/edition_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/edition.csv
   --relationships=EDCOMP_ATH=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/edcomp_ath_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/edcomp_ath.csv
   --relationships=ATHPART_ED=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/athpart_ed_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/athpart_ed.csv   
   --nodes=COUNTRY=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/country_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/country.csv 
   --relationships=BORNED_IN=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/borned_in_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/borned_in.csv
   --relationships=REPRESENT_TO=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/represent_to_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/represent_to.csv 
   --nodes=CITY=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/city_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/city.csv
   --relationships=ORGANIZED_BY=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/organized_by_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/organized_by.csv
   --relationships=LOCATED_IN=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/located_in_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/located_in.csv 
   --nodes=RESULT=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/result_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/result.csv
   --relationships=ORGANIZED_BY=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/performed_in_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/performed_in.csv 
   --nodes=RESULT=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/medal_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/medal.csv
   --nodes=EVENT=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/event_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/event.csv
   --relationships=COMPETED_IN=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/competed_in_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/competed_in.csv 
   --nodes=SPORT=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/sport_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/sport.csv
   --relationships=COMPETED_IN=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/wins_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/wins.csv 
   --relationships=COMPETED_IN=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/sp_cpofevt_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/sp_cpofevt.csv 
   --relationships=COMPETED_IN=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/ed_orgzevt_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/ed_orgzevt.csv 
--trim-strings=true --multiline-fields=true --overwrite-destination neo4j  --verbose
'''

'\nbin/neo4j-admin database import full --delimiter="," --quote="\'" \n   --nodes=ATHLETE=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/athlete_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/athlete.csv \n   --nodes=EDITION=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/edition_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/edition.csv\n   --relationships=EDCOMP_ATH=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/edcomp_ath_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/edcomp_ath.csv\n   --relationships=ATHPART_ED=/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/athpart_ed_header.csv, /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/fichiers/athpart_ed.csv   \n   --nodes=COUNTRY=

Voir tous les athlètes   

    - MATCH (a:Athlete) RETURN a LIMIT 25   

Voir toutes les éditions  

    - MATCH (e:Edition) RETURN e LIMIT 25

Voir tous les sports   

    - MATCH (s:Sport) RETURN s LIMIT 25

Voir tous les types de nœuds et relations   

    - MATCH (n)  
    - RETURN DISTINCT labels(n), count(*) as count

Voir le nombre de relations par type

    - MATCH ()-[r]->() 
    - RETURN DISTINCT type(r), count(*) as count

   1) Donner le nombre de nœuds par label ;  

    - MATCH (n)    
      - WITH labels(n) as labels
      - UNWIND labels as label
      - RETURN DISTINCT label, count(*) as count
      - ORDER BY label   

ou     

    - CALL apoc.meta.stats()
      - YIELD labels
      - RETURN labels  

   2) Donner le nombre de relations par type ;

    - MATCH ()-[r]->()
      - RETURN type(r) as relationType, count(*) as count
      - ORDER BY count DESC 

ou

    - CALL apoc.meta.stats()
      - YIELD relTypesCount
      - RETURN relTypesCount  


   3) Donner les athlètes (nom, pays représenté) qui ont gagné une médaille à l’épreuve « Decathlon, Men » en 2020 ;

    - MATCH (a:Athlete)-[r:COMPETED_IN]->(ed:Edition),
      - (a)-[:REPRESENTS]->(c:Country)
    - WHERE ed.edition = "2020 Summer Olympics" 
      - AND r.event = "Decathlon, Men"
      - AND r.medal IN ["GOLD", "SILVER", "BRONZE"]
    - RETURN a.name as Athlete, c.country as Country, r.medal as Medal
    - ORDER BY CASE r.medal 
      - WHEN "GOLD" THEN 1 
      - WHEN "SILVER" THEN 2 
      - WHEN "BRONZE" THEN 3 
    - END


   4) Donner le nombre d’athlètes féminines en 2016 ;

    - MATCH (a:Athlete)-[r:COMPETED_IN]->(e:Edition)
    - WHERE e.edition = "2016 Summer Olympics" 
        - AND r.sex = "F"
    - RETURN COUNT(DISTINCT a) as NombreAthletesFeminines  

ou    

    - CALL apoc.meta.stats()
    - YIELD nodeCount
    - WITH nodeCount
    - MATCH (a:Athlete)-[r:COMPETED_IN]->(e:Edition)
    - WHERE e.edition = "2016 Summer Olympics" 
    - AND r.sex = "F"
    - RETURN count(DISTINCT a) as NombreAthletesFeminines

   5) Donner tous les athlètes qui ont participé aux jeux pour un pays dans lequel ils ne sont pas nés ;

    - MATCH (a:Athlete)-[r:COMPETED_IN]->(e:Edition)
    - MATCH (a)-[:BORN_IN]->(birth:Country)
    - MATCH (a)-[:REPRESENTS]->(comp:Country)
    - WHERE birth <> comp
    - RETURN DISTINCT a.name as Athlete, 
      - birth.country as PaysDeNaissance, 
      - comp.country as PaysRepresente
    - ORDER BY a.name   

ou

    - CALL apoc.cypher.run("
    - MATCH (a:Athlete)-[r:COMPETED_IN]->(e:Edition)
    - MATCH (a)-[:BORN_IN]->(birth:Country)
    - MATCH (a)-[:REPRESENTS]->(comp:Country)
    - WHERE birth <> comp
    - RETURN DISTINCT a.name as Athlete, 
         - birth.country as PaysDeNaissance, 
         - comp.country as PaysRepresente
    - ORDER BY a.name
    - ", {}) YIELD value
    - RETURN value.Athlete as Athlete,
        - value.PaysDeNaissance as PaysDeNaissance,
        - value.PaysRepresente as PaysRepresente 


   6) Donner les tweets de l’édition 2020 qui concernent le nageur Michael Phelps (hashtag michaelphelps) ;

   7) Donner les disciplines (et les sports associés) qui ont été proposées sur moins de 10 éditions.
 

In [26]:
"""
#olympics_result    = pd.read_csv(path_prep + 'Olympics_Games_prepared.csv')
olympics_games  = pd.read_csv(path_prep + 'Olympic_Results_Standardized.csv')
distinct_results = olympics_games[['result_id','event_title']].drop_duplicates().reset_index(drop=True)
result_id_map     = {event_title: result_id for event_title, result_id in distinct_results.iterrows()}

distinctresults  = olympics_games[['result_id','event_title','sport','edition_id','result_date','result_location']].drop_duplicates().reset_index(drop=True)
print("Doublons dans Olympic_Results_Standardized:", len(distinct_results))
print("Doublons dans Olympic_Results_Standardized:", len(distinctresults))
print("Doublons dans Olympic_Results_Standardized:", distinct_results.duplicated().sum())
print("Doublons dans Olympic_Results_Standardized:", distinctresults.duplicated().sum())

for value in result_id_map.values():
    print(f"result_id: {value['result_id']}")
    print(f"event_title: {value['event_title']}")
    """


'\n#olympics_result    = pd.read_csv(path_prep + \'Olympics_Games_prepared.csv\')\nolympics_games  = pd.read_csv(path_prep + \'Olympic_Results_Standardized.csv\')\ndistinct_results = olympics_games[[\'result_id\',\'event_title\']].drop_duplicates().reset_index(drop=True)\nresult_id_map     = {event_title: result_id for event_title, result_id in distinct_results.iterrows()}\n\ndistinctresults  = olympics_games[[\'result_id\',\'event_title\',\'sport\',\'edition_id\',\'result_date\',\'result_location\']].drop_duplicates().reset_index(drop=True)\nprint("Doublons dans Olympic_Results_Standardized:", len(distinct_results))\nprint("Doublons dans Olympic_Results_Standardized:", len(distinctresults))\nprint("Doublons dans Olympic_Results_Standardized:", distinct_results.duplicated().sum())\nprint("Doublons dans Olympic_Results_Standardized:", distinctresults.duplicated().sum())\n\nfor value in result_id_map.values():\n    print(f"result_id: {value[\'result_id\']}")\n    print(f"event_title: {va

In [27]:
"""    
# Créer les dictionnaires de correspondance
olympics_country   = pd.read_csv(path_prep + 'Olympics_Country_Standardized.csv')
distinct_country   = olympics_country[['noc','country']].drop_duplicates().reset_index(drop=True)

country_id_map     = {noc : country for noc, country in distinct_country.iterrows()}

# Écrire les données des pays
for value in country_id_map.values():
     print(f"{value['noc']},{value['country'].strip(',')},COUNTRY\n")
     """

'    \n# Créer les dictionnaires de correspondance\nolympics_country   = pd.read_csv(path_prep + \'Olympics_Country_Standardized.csv\')\ndistinct_country   = olympics_country[[\'noc\',\'country\']].drop_duplicates().reset_index(drop=True)\n\ncountry_id_map     = {noc : country for noc, country in distinct_country.iterrows()}\n\n# Écrire les données des pays\nfor value in country_id_map.values():\n     print(f"{value[\'noc\']},{value[\'country\'].strip(\',\')},COUNTRY\n")\n     '

In [28]:
"""    
# Créer les dictionnaires de correspondance
olympics_games    = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')
distinct_country  = olympics_games[['country_noc','city']].drop_duplicates().reset_index(drop=True)
distinctcountry   = olympics_games[['city','edition_id']].drop_duplicates().reset_index(drop=True)
print("Doublons dans distinct_country:", len(distinct_country))
print("Doublons dans distinctcountry:", len(distinctcountry))
print("Doublons dans distinct_country:", distinct_country.duplicated().sum())
print("Doublons dans distinctcountry:", distinctcountry.duplicated().sum())

city_country_map  = {country_noc : city for country_noc, city in distinct_country.iterrows()}

# Écrire les données des pays
for value in city_country_map.values():
     print(f"{value['country_noc']},{value['city'].strip(',')},COUNTRY\n")
     """

'    \n# Créer les dictionnaires de correspondance\nolympics_games    = pd.read_csv(path_prep + \'Olympics_Games_Standardized.csv\')\ndistinct_country  = olympics_games[[\'country_noc\',\'city\']].drop_duplicates().reset_index(drop=True)\ndistinctcountry   = olympics_games[[\'city\',\'edition_id\']].drop_duplicates().reset_index(drop=True)\nprint("Doublons dans distinct_country:", len(distinct_country))\nprint("Doublons dans distinctcountry:", len(distinctcountry))\nprint("Doublons dans distinct_country:", distinct_country.duplicated().sum())\nprint("Doublons dans distinctcountry:", distinctcountry.duplicated().sum())\n\ncity_country_map  = {country_noc : city for country_noc, city in distinct_country.iterrows()}\n\n# Écrire les données des pays\nfor value in city_country_map.values():\n     print(f"{value[\'country_noc\']},{value[\'city\'].strip(\',\')},COUNTRY\n")\n     '

In [29]:
"""
olympics_games     = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')
distinct_cty_edt = olympics_games[['city','edition_id']].drop_duplicates().reset_index(drop=True)
 # Écrire les données des pays
for _, value in distinct_cty_edt.iterrows():
     print(f"{ value['city']},{value['edition_id']},COUNTRY\n")
     """

'\nolympics_games     = pd.read_csv(path_prep + \'Olympics_Games_Standardized.csv\')\ndistinct_cty_edt = olympics_games[[\'city\',\'edition_id\']].drop_duplicates().reset_index(drop=True)\n # Écrire les données des pays\nfor _, value in distinct_cty_edt.iterrows():\n     print(f"{ value[\'city\']},{value[\'edition_id\']},COUNTRY\n")\n     '

In [30]:
import re

#olympic_results   = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')
#olympics_res_Std  = pd.read_csv(path_prep + 'Olympic_Results_Standardized.csv')
#olympics_rt_prep  = pd.read_csv(path_prep + 'Olympics_Games_prepared.csv')
#athlete_results    = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv')

#distinct_athletes = athlete_results[['athlete_id', 'name','sex','country_noc']].drop_duplicates(subset=['athlete_id'], keep='first').reset_index(drop=True)  
#distinctathletes = athlete_results[['athlete_id', 'name','sex']].drop_duplicates(subset=['athlete_id'], keep='first').reset_index(drop=True)  
#print("len dans Olympic_Results_Standardized distinct_athletes:", len(distinct_athletes)) 
#print("len dans Olympic_Results_Standardized distinctathletes:", len(distinctathletes)) 

#olympics_games    = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')
# ['athlete_id', 'event', 'result_id']
#distinct_1 = olympics_rt_prep[['edition_id', 'country_noc', 'athlete_id','event_title','result_id']].drop_duplicates().reset_index(drop=True)
#distinct_2 = olympics_rt_prep[['edition_id','city','country_noc']].drop_duplicates().reset_index(drop=True)
#distinct_3 = olympics_rt_prep[['edition_id','city']].drop_duplicates().reset_index(drop=True)
#distinct_4 = olympics_rt_prep[['edition_id']].drop_duplicates().reset_index(drop=True)
#print(f"len dans complete_result: {len(distinct_1)}, {len(distinct_2)}, {len(distinct_3)}, {len(distinct_4)}")

#distinct_events  = olympics_res_Std[['result_id','event_title','edition_id','sport']].drop_duplicates().reset_index(drop=True)
#distinctsresult = olympics_res_Std[['result_id','event_title','edition_id']].drop_duplicates().reset_index(drop=True)
#distincts_result = olympics_res_Std[['result_id']].drop_duplicates().reset_index(drop=True)
#event_sport_map  = distinct_events.drop_duplicates('event_title').set_index('event_title')['sport'].to_dict()
#sport_event_map  = distinct_events.drop_duplicates('sport').set_index('event_title')['sport'].to_dict()

#print("len dans Olympic_Results_Standardized distinct_events:", len(distinct_events))
#print("len dans Olympic_Results_Standardized distinctsresult:", len(distinctsresult))
#print("len dans Olympic_Results_Standardized distinctsresult:", len(distincts_result))
#print("len dans Olympic_Results_Standardized event_sport_map:", len(event_sport_map))
#print("len dans Olympic_Results_Standardized sport_event_map:", len(sport_event_map))

#distincts_prep   = olympics_rt_prep[['result_id','event_title','sport','edition_id','result_date','result_location']].drop_duplicates().reset_index(drop=True)
#distincts_result = olympics_rt_prep[['result_id','country_noc','event_title','sport','edition_id','result_date','result_location']].drop_duplicates().reset_index(drop=True)

#complete_result  = olympic_results[['result_id','athlete_id']].drop_duplicates().reset_index(drop=True)
#distinct_result  = olympic_results[['result_id','event']].drop_duplicates().reset_index(drop=True)
#distinct_event   = olympic_results['event'].drop_duplicates().reset_index(drop=True)
#distinct_edition = olympic_results['result_id'].drop_duplicates().reset_index(drop=True)

"""
print("len dans Olympics_Games_prepared distincts_prep :", len(distincts_prep))
print("len dans Olympics_Games_prepared ---  distincts_result:", len(distincts_result))

print("len dans Olympic_Athlete_Event_Results_Standardized complete_result :", len(complete_result))
print("len dans Olympic_Athlete_Event_Results_Standardized distinct_result :", len(distinct_result))
print("len dans Olympic_Athlete_Event_Results_Standardized distinct_event  :", len(distinct_event))
print("len dans Olympic_Athlete_Event_Results_Standardized distinct_edition:", len(distinct_edition))

print("Doublons dans distincts_result:", distincts_result.duplicated().sum())
print("Doublons dans distincts_prep:", distincts_prep.duplicated().sum())
print("Doublons dans distincts_result:", distincts_result.duplicated().sum())

print("Doublons dans complete_result:", complete_result.duplicated().sum())
print("Doublons dans distinct_result:", distinct_result.duplicated().sum())
print("Doublons dans distinct_event:", distinct_event.duplicated().sum())
print("Doublons dans distinct_edition:", distinct_edition.duplicated().sum())
"""

'''
complete_result = olympic_results[['result_id','edition_id','athlete_id','sport','event']].drop_duplicates().reset_index(drop=True)
event_sport_map  = complete_result.drop_duplicates('event').set_index('event')['sport'].to_dict()
#event_sport_map
 # Écrire les données des pays
for key, value in event_sport_map.items():
 #     gender  = re.search(r'\b(Men|Women)\b', key).group(1)
      print(f" -  {key} : {value},COUNTRY\n")
      #print(f" - { value['event']}, - {value['sport']},COUNTRY\n")
      '''

'\ncomplete_result = olympic_results[[\'result_id\',\'edition_id\',\'athlete_id\',\'sport\',\'event\']].drop_duplicates().reset_index(drop=True)\nevent_sport_map  = complete_result.drop_duplicates(\'event\').set_index(\'event\')[\'sport\'].to_dict()\n#event_sport_map\n # Écrire les données des pays\nfor key, value in event_sport_map.items():\n #     gender  = re.search(r\'\x08(Men|Women)\x08\', key).group(1)\n      print(f" -  {key} : {value},COUNTRY\n")\n      #print(f" - { value[\'event\']}, - {value[\'sport\']},COUNTRY\n")\n      '

In [31]:

"""
# Charger les fichiers CSV
athlete_events = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')
athlete_bio = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv')
results = pd.read_csv(path_prep + 'Olympic_Results_Standardized.csv')
olympic_res  = pd.read_csv(path_prep + 'Olympics_Games_prepared.csv')

#distinct_athl = athlete_bio['athlete_id'].drop_duplicates().reset_index(drop=True)
#print(len(distinct_athl))

edition = pd.read_csv(path_load + "fichiers/" + 'edition.csv')
print("edition : ", edition)

edition = olympic_res['edition_id'].drop_duplicates().reset_index(drop=True)
print("olympic_res : ", edition)


edition = athlete_events['edition_id'].drop_duplicates().reset_index(drop=True)
print("athlete_events : ",edition)

edition = results['edition_id'].drop_duplicates().reset_index(drop=True)
print("results : ", edition)

edition = olympic_res['edition_id'].drop_duplicates().reset_index(drop=True)
print("olympic_res : ", edition)

#olympic_res  = pd.read_csv(path_prep + 'Olympics_Games_prepared.csv')

#distinct_athlete = olympic_res['athlete_id'].drop_duplicates().reset_index(drop=True)
#print(len(distinct_athlete))

#olympic_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')
#print(len(olympic_results))

#olympic_res  = pd.read_csv(path_prep + 'Olympics_Games_prepared.csv')
#print(len(olympic_res))


#olympics_res_Std = pd.read_csv(path_prep + 'Olympics_Games_prepared.csv')
#distinct_events = olympics_res_Std.drop_duplicates('result_id').reset_index(drop=True) 

#print("distincts events" ,len(distinct_events))


#distinct_sports  = olympics_res_Std['sport'].drop_duplicates().reset_index(drop=True).str.strip().str.replace(',', '.') 
#sport_id_map     = {sport: f'sp{i+1}' for i, sport in enumerate(distinct_sports)}
#event_id_map     = {row['result_id']: f'ev{i+1}' for i, row in distinct_events.iterrows()}
#event_sport_map  = distinct_events.drop_duplicates('event_title').set_index('event_title')['sport'].to_dict()
"""


'\n# Charger les fichiers CSV\nathlete_events = pd.read_csv(path_prep + \'Olympic_Athlete_Event_Results_Standardized.csv\')\nathlete_bio = pd.read_csv(path_prep + \'Olympic_Athlete_Bio_Standardized.csv\')\nresults = pd.read_csv(path_prep + \'Olympic_Results_Standardized.csv\')\nolympic_res  = pd.read_csv(path_prep + \'Olympics_Games_prepared.csv\')\n\n#distinct_athl = athlete_bio[\'athlete_id\'].drop_duplicates().reset_index(drop=True)\n#print(len(distinct_athl))\n\nedition = pd.read_csv(path_load + "fichiers/" + \'edition.csv\')\nprint("edition : ", edition)\n\nedition = olympic_res[\'edition_id\'].drop_duplicates().reset_index(drop=True)\nprint("olympic_res : ", edition)\n\n\nedition = athlete_events[\'edition_id\'].drop_duplicates().reset_index(drop=True)\nprint("athlete_events : ",edition)\n\nedition = results[\'edition_id\'].drop_duplicates().reset_index(drop=True)\nprint("results : ", edition)\n\nedition = olympic_res[\'edition_id\'].drop_duplicates().reset_index(drop=True)\nprin

In [32]:
"""
olympics_games     = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')

distinct_edition      = olympics_games['edition_id'].drop_duplicates().reset_index(drop=True)
print(len(distinct_edition))

distinct_city      = olympics_games['city'].drop_duplicates().reset_index(drop=True)
city_id_map        = {value: f'ci{i+1}' for i, value in distinct_city.items()}
print(len(distinct_city))
print(len(city_id_map))

distinct_cted      = olympics_games[['city','edition_id']].drop_duplicates().reset_index(drop=True)
city_edID_map      = { value['edition_id'] : value['city'] for _, value in distinct_cted.iterrows()}
print(len(distinct_cted))
print(len(city_edID_map))

distinct_edyr      = olympics_games[['edition_id','year']].drop_duplicates().reset_index(drop=True)
year_edID_map      = {value['edition_id'] : value['year'] for _, value in distinct_edyr.iterrows()}
print(len(distinct_edyr))
print(len(year_edID_map))

distinct_cityCntry = olympics_games[['edition_id','country_noc']].drop_duplicates().reset_index(drop=True)
print(len(distinct_cityCntry))
"""


"\nolympics_games     = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')\n\ndistinct_edition      = olympics_games['edition_id'].drop_duplicates().reset_index(drop=True)\nprint(len(distinct_edition))\n\ndistinct_city      = olympics_games['city'].drop_duplicates().reset_index(drop=True)\ncity_id_map        = {value: f'ci{i+1}' for i, value in distinct_city.items()}\nprint(len(distinct_city))\nprint(len(city_id_map))\n\ndistinct_cted      = olympics_games[['city','edition_id']].drop_duplicates().reset_index(drop=True)\ncity_edID_map      = { value['edition_id'] : value['city'] for _, value in distinct_cted.iterrows()}\nprint(len(distinct_cted))\nprint(len(city_edID_map))\n\ndistinct_edyr      = olympics_games[['edition_id','year']].drop_duplicates().reset_index(drop=True)\nyear_edID_map      = {value['edition_id'] : value['year'] for _, value in distinct_edyr.iterrows()}\nprint(len(distinct_edyr))\nprint(len(year_edID_map))\n\ndistinct_cityCntry = olympics_games[['edition_id',

In [33]:
#athlete_results   = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv')
#olympic_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')

#distinct_athletes1 = athlete_results[['athlete_id', 'name','sex','country_noc']].drop_duplicates(subset=['athlete_id'], keep='first').reset_index(drop=True)  
#distinct_athletes2 = athlete_results['athlete_id'].drop_duplicates( keep='first').reset_index(drop=True)  
#distinct_athletes3 = olympic_results['athlete_id'].drop_duplicates( keep='first').reset_index(drop=True)  

#distinct_athletes3 = olympic_results[['athlete_id', 'athlete','country_noc']].drop_duplicates( keep='first').reset_index(drop=True)  

#print(f'{len(distinct_athletes1)}')
#print(f'{len(distinct_athletes2)}')

#print(f'{len(distinct_athletes3)}')


In [34]:

#difference = set(distinct_athletes3) - set(distinct_athletes2)
# Créez un masque pour les athlètes supplémentaires
#mask = olympic_results['athlete_id'].isin(difference)
# Sélectionnez les lignes correspondantes dans olympic_results

#additional_athletes = olympic_results[mask].drop_duplicates('athlete_id')[['athlete_id', 'country_noc', 'athlete']]
# Ajoutez ces lignes à athlete_results
#athlete_results = pd.concat([athlete_results, additional_athletes], ignore_index=True)


In [35]:
"""
olympic_results   = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv')
athlete_results   = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Prepared.csv')

distinct_olympic = athlete_results['athlete_id'].drop_duplicates( keep='first').reset_index(drop=True)  
print(len(distinct_olympic))
distinct_biorest = olympic_results['athlete_id'].drop_duplicates( keep='first').reset_index(drop=True)  
print(len(distinct_biorest))

difference = set(distinct_biorest) - set(distinct_olympic)
print(len(difference))
# Créez un masque pour les athlètes supplémentaires

mask = olympic_results['athlete_id'].isin(difference)

# Sélectionnez les lignes correspondantes dans olympic_results
additional_athletes = olympic_results[mask].drop_duplicates('athlete_id')[['athlete_id', 'athlete']]
additional_athletes = additional_athletes.rename(columns={'athlete': 'name'})
additional_athletes['sex'] = 'unknown'
additional_athletes['country_noc'] = 'unknown'



for _, athlete in additional_athletes.iterrows():
      print(f"{athlete['athlete_id']} {athlete['athlete_id']}")

# Ajoutez ces lignes à athlete_results
#athlete_results = pd.concat([athlete_results, additional_athletes], ignore_index=True, axis=0)

#distinct_athletes = athlete_results[['athlete_id', 'name','sex','country_noc']].drop_duplicates(subset=['athlete_id'], keep='first').reset_index(drop=True)  
#distinct_athletes
"""

'\nolympic_results   = pd.read_csv(path_prep + \'Olympic_Athlete_Event_Results_Prepared.csv\')\nathlete_results   = pd.read_csv(path_prep + \'Olympic_Athlete_Bio_Prepared.csv\')\n\ndistinct_olympic = athlete_results[\'athlete_id\'].drop_duplicates( keep=\'first\').reset_index(drop=True)  \nprint(len(distinct_olympic))\ndistinct_biorest = olympic_results[\'athlete_id\'].drop_duplicates( keep=\'first\').reset_index(drop=True)  \nprint(len(distinct_biorest))\n\ndifference = set(distinct_biorest) - set(distinct_olympic)\nprint(len(difference))\n# Créez un masque pour les athlètes supplémentaires\n\nmask = olympic_results[\'athlete_id\'].isin(difference)\n\n# Sélectionnez les lignes correspondantes dans olympic_results\nadditional_athletes = olympic_results[mask].drop_duplicates(\'athlete_id\')[[\'athlete_id\', \'athlete\']]\nadditional_athletes = additional_athletes.rename(columns={\'athlete\': \'name\'})\nadditional_athletes[\'sex\'] = \'unknown\'\nadditional_athletes[\'country_noc\'] = \

In [36]:
"""
olympic_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')
athlete_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv')

distinct_olympic = athlete_results['athlete_id'].drop_duplicates( keep='first').reset_index(drop=True)  
distinct_biorest = olympic_results['athlete_id'].drop_duplicates( keep='first').reset_index(drop=True)  

difference = set(distinct_biorest) - set(distinct_olympic)
# Créez un masque pour les athlètes supplémentaires
mask = olympic_results['athlete_id'].isin(difference)
# Sélectionnez les lignes correspondantes dans olympic_results
additional_athletes = olympic_results[mask].drop_duplicates('athlete_id')[['athlete_id', 'athlete']]
additional_athletes = additional_athletes.rename(columns={'athlete': 'name'})
additional_athletes['sex'] = 'unknown'
additional_athletes['country_noc'] = 'UNK'
additional_athletes['country'] = 'unknown'
additional_athletes['born'] = 'unknown'
# Ajoutez ces lignes à athlete_results
athlete_results = pd.concat([athlete_results, additional_athletes], ignore_index=True)
# Sauvegarder dans un nouveau fichier
athlete_results.to_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv', index=False)
"""

"\nolympic_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')\nathlete_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv')\n\ndistinct_olympic = athlete_results['athlete_id'].drop_duplicates( keep='first').reset_index(drop=True)  \ndistinct_biorest = olympic_results['athlete_id'].drop_duplicates( keep='first').reset_index(drop=True)  \n\ndifference = set(distinct_biorest) - set(distinct_olympic)\n# Créez un masque pour les athlètes supplémentaires\nmask = olympic_results['athlete_id'].isin(difference)\n# Sélectionnez les lignes correspondantes dans olympic_results\nadditional_athletes = olympic_results[mask].drop_duplicates('athlete_id')[['athlete_id', 'athlete']]\nadditional_athletes = additional_athletes.rename(columns={'athlete': 'name'})\nadditional_athletes['sex'] = 'unknown'\nadditional_athletes['country_noc'] = 'UNK'\nadditional_athletes['country'] = 'unknown'\nadditional_athletes['born'] = 'unknown'\n# Ajoutez ces lignes 

In [37]:
"""
olympics_res_Std = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv') 
#distinct_events  = olympics_res_Std.drop_duplicates('result_id').reset_index(drop=True)
#distinctEvents   = distinct_events.drop_duplicates('event').applymap(lambda x: x.strip().replace(',', '.') if isinstance(x, str) else x)
#event_sport_map   =  {row['event']: row['sport'] for i, row in distinctEvents.iterrows()}
distinct_sports   = olympics_res_Std.drop_duplicates('sport').apply(lambda col: col.map(lambda x: x.strip().replace(',', '.') if isinstance(x, str) else x))
sport_id_map      = {row['sport']: f'sp{i+1}' for i, row in distinct_sports.iterrows()}
#distinct_sports  = olympics_res_Std['sport'].drop_duplicates().reset_index(drop=True).str.strip().str.replace(',', '.') 
sport_id_map
#distinctEvents
"""

"\nolympics_res_Std = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv') \n#distinct_events  = olympics_res_Std.drop_duplicates('result_id').reset_index(drop=True)\n#distinctEvents   = distinct_events.drop_duplicates('event').applymap(lambda x: x.strip().replace(',', '.') if isinstance(x, str) else x)\n#event_sport_map   =  {row['event']: row['sport'] for i, row in distinctEvents.iterrows()}\ndistinct_sports   = olympics_res_Std.drop_duplicates('sport').apply(lambda col: col.map(lambda x: x.strip().replace(',', '.') if isinstance(x, str) else x))\nsport_id_map      = {row['sport']: f'sp{i+1}' for i, row in distinct_sports.iterrows()}\n#distinct_sports  = olympics_res_Std['sport'].drop_duplicates().reset_index(drop=True).str.strip().str.replace(',', '.') \nsport_id_map\n#distinctEvents\n"

In [38]:
"""
# Définit le noeud et les données CITY et la relation LOCATED_IN entre CITY et COUNTRY
def process_city(distinct_city):    
    with \
        open(path_load + "fichiers/"  + 'city.csv', 'w', newline='') as _city_file:

        _city_file.write(f'city_id:ID,city, :LABEL\n')
        for _, value in distinct_city.items():
            city_id     = city_id_map.get(value)
            _city_file.write(f'{city_id}, {value}, CITY\n')

olympics_games     = pd.read_csv(path_prep + 'Olympics_Games_Standardized.csv')
distinct_city      = olympics_games['city'].drop_duplicates().reset_index(drop=True)
city_id_map        = {value: f'ci{i+1}' for i, value in distinct_city.items()}

#process_city(distinct_city)
"""

'\n# Définit le noeud et les données CITY et la relation LOCATED_IN entre CITY et COUNTRY\ndef process_city(distinct_city):    \n    with         open(path_load + "fichiers/"  + \'city.csv\', \'w\', newline=\'\') as _city_file:\n\n        _city_file.write(f\'city_id:ID,city, :LABEL\n\')\n        for _, value in distinct_city.items():\n            city_id     = city_id_map.get(value)\n            _city_file.write(f\'{city_id}, {value}, CITY\n\')\n\nolympics_games     = pd.read_csv(path_prep + \'Olympics_Games_Standardized.csv\')\ndistinct_city      = olympics_games[\'city\'].drop_duplicates().reset_index(drop=True)\ncity_id_map        = {value: f\'ci{i+1}\' for i, value in distinct_city.items()}\n\n#process_city(distinct_city)\n'